---
jupyter: python3
execute:
  echo: false
  output: asis
lightbox: 
  match: auto
  effect: fade
  desc-position: bottom
  loop: true
---

In [108]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pprint 
from pathlib import Path
from collections import defaultdict

def query_WB(endpoint, query):

    # Initialize the wrapper
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    # Fetch and parse results
    results = sparql.query().convert()

    return results 

def get_data(results):
    
    return [
    {
        "url":           result["item"]["value"],
        "label":         result["itemLabel"]["value"],
        "description":   result["itemDescription"]["value"],
        "photo":         result.get("photo",{}).get("value",""),
        "roomLink":      result.get("parent",{}).get("value",""),
        "room":          result.get("parentLabel",{}).get("value",""),
        "type":          result["itemTypeLabel"]["value"],
        "creator":       result.get("creator",{}).get("value","-"),
    }
    for result in results["results"]["bindings"]
  ]

def is_supported_image(path):
    ext = Path(path).suffix.lower()
    return ext in [".jpg", ".jpeg", ".png"]

def generate_output(data,castle):
    
    print(f""" 
# Ausstellung: {castle}
    """)

    for entry in data: 
      if not is_supported_image(entry["photo"]):
        continue
      else:
        print(f"""
              
![{entry["label"]}. Foto: {entry["creator"]}]({entry["photo"]}){{group="photos"}}

\\newpage
        """)





In [109]:
castle_ID = "Q68"

In [110]:
def make_book():

    endpoint_url = "https://query.kewl.org/sparql"

    query = f"""
PREFIX wd: <https://wikibase.kewl.org/entity/>
PREFIX wdt: <https://wikibase.kewl.org/prop/direct/>
PREFIX p: <https://wikibase.kewl.org/prop/>
PREFIX ps: <https://wikibase.kewl.org/prop/statement/>
PREFIX pq: <https://wikibase.kewl.org/prop/qualifier/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX bd: <http://www.bigdata.com/rdf#>


SELECT DISTINCT ?item ?itemLabel ?itemDescription ?itemTypeLabel ?photo ?creator ?parent ?parentLabel ?castleLabel

WHERE {{
  
  VALUES ?itemType {{ wd:Q6 }}
  
  ?item wdt:P3+ wd:{castle_ID} ;
        wdt:P1 ?itemType .
  OPTIONAL {{ ?item wdt:P3 ?parent . }}
  OPTIONAL {{
    ?item p:P6 ?statement .
    ?statement ps:P6 ?photo .
    OPTIONAL {{ ?statement pq:P11 ?creator. }}
  }}
  BIND(wd:{castle_ID} AS ?castle)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "de" }}
}}
ORDER BY ?item
    """


    results = query_WB(endpoint_url,query)
    castle = next((r["castleLabel"]["value"] for r in results["results"]["bindings"] if "castleLabel" in r))
    data = get_data(results)
    generate_output(data,castle)

make_book()

 
# Ausstellung: Stuttgart, Schloss Solitude
    


![Wanddekoration mit Blumengirlanden. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013272a.jpg){group="photos"}

\newpage
        


![Auferstehung Christi. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013274a.jpg){group="photos"}

\newpage
        


![Auferstehung Christi. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013273a.jpg){group="photos"}

\newpage
        


![Moses mit den Gesetzestafeln. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013277a.jpg){group="photos"}

\newpage
        


![Ruinenlandschaften. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013289a.jpg){group="photos"}

\newpage
        


![Wanddekoration mit Blumengirlanden. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013290a.jpg){group="photos"}

\newpage
        


![Zwickelfelder der Voute. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013283a.jpg){gro

## [Wikibase Query Link](https://query.kewl.org/#PREFIX%20wd%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fentity%2F%3E%0APREFIX%20wdt%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2Fdirect%2F%3E%0APREFIX%20p%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2F%3E%0APREFIX%20ps%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2Fstatement%2F%3E%0APREFIX%20pq%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2Fqualifier%2F%3E%0APREFIX%20wikibase%3A%20%3Chttp%3A%2F%2Fwikiba.se%2Fontology%23%3E%0APREFIX%20rdfs%3A%20%3Chttp%3A%2F%2Fwww.w3.org%2F2000%2F01%2Frdf-schema%23%3E%0APREFIX%20bd%3A%20%3Chttp%3A%2F%2Fwww.bigdata.com%2Frdf%23%3E%0A%0A%0ASELECT%20DISTINCT%20%3Fitem%20%3FitemLabel%20%3FitemDescription%20%3FitemTypeLabel%20%3Fphoto%20%3Fcreator%20%3Fparent%20%3FparentLabel%20%3FcastleLabel%0A%0AWHERE%20%7B%0A%20%20%0A%20%20VALUES%20%3FitemType%20%7B%20wd%3AQ6%20%7D%0A%20%20%0A%20%20%3Fitem%20wdt%3AP3%2B%20wd%3AQ68%20%3B%0A%20%20%20%20%20%20%20%20wdt%3AP1%20%3FitemType%20.%0A%20%20OPTIONAL%20%7B%20%3Fitem%20wdt%3AP3%20%3Fparent%20.%20%7D%0A%20%20OPTIONAL%20%7B%0A%20%20%20%20%3Fitem%20p%3AP6%20%3Fstatement%20.%0A%20%20%20%20%3Fstatement%20ps%3AP6%20%3Fphoto%20.%0A%20%20%20%20OPTIONAL%20%7B%20%3Fstatement%20pq%3AP11%20%3Fcreator.%20%7D%0A%20%20%7D%0A%20%20BIND%28wd%3AQ68%20AS%20%3Fcastle%29%0A%20%20SERVICE%20wikibase%3Alabel%20%7B%20bd%3AserviceParam%20wikibase%3Alanguage%20%22de%22%20%7D%0A%7D%0AORDER%20BY%20%3Fitem)